# 🎓 Student Placement Probability Predictor
## ML Training Notebook

**Pipeline:**
1. Data Loading & EDA
2. Feature Engineering & Preprocessing
3. Model Training — XGBoost (Placement Probability) + Random Forest (Company Tier)
4. Model Evaluation — Accuracy, F1, ROC-AUC, Confusion Matrix
5. SHAP Explainability — Feature Impact & Gap Analysis
6. Save Models for FastAPI

**Target Variables:**
- `placed` → Binary (0/1) — XGBoost probability output
- `company_tier` → Multi-class (FAANG / Mid-tier / Mass Recruiter / Not Placed) — Random Forest

---
## 📦 Cell 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn imbalanced-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import joblib
import os

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

# ML
from sklearn.model_selection      import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing        import LabelEncoder, StandardScaler
from sklearn.metrics               import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble              import RandomForestClassifier
from sklearn.linear_model          import LogisticRegression
from imblearn.over_sampling        import SMOTE
import xgboost as xgb
import shap

print('✅ All libraries imported successfully')
print(f'   XGBoost  : {xgb.__version__}')
print(f'   SHAP     : {shap.__version__}')
print(f'   sklearn  : {__import__("sklearn").__version__}')

---
## 📂 Cell 2 — Load Dataset

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────
# Update path if needed
DATA_PATH = 'student_placement_10k.csv'

df = pd.read_csv(DATA_PATH)

print(f'Shape         : {df.shape}')
print(f'Null values   : {df.isnull().sum().sum()}')
print(f'Columns       : {list(df.columns)}')
print()
df.head()

In [ ]:
# Data types & basic stats
df.describe().round(2)

---
## 📊 Cell 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1  Target distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Placement binary
placed_counts = df['placed'].value_counts()
axes[0].bar(['Not Placed', 'Placed'], placed_counts.values,
            color=['#ef4444', '#22c55e'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Placed vs Not Placed', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(placed_counts.values):
    axes[0].text(i, v + 30, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

# Company tier
tier_order  = ['FAANG', 'Mid-tier', 'Mass Recruiter', 'Not Placed']
tier_colors = ['#6366f1', '#3b82f6', '#10b981', '#ef4444']
tier_counts = df['company_tier'].value_counts().reindex(tier_order)
bars = axes[1].barh(tier_order, tier_counts.values, color=tier_colors, edgecolor='white')
axes[1].set_title('Company Tier Distribution', fontweight='bold')
axes[1].set_xlabel('Count')
for i, v in enumerate(tier_counts.values):
    axes[1].text(v + 20, i, f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('plot_target_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: plot_target_distribution.png')

In [ ]:
# ── 3.2  Feature distributions by placement tier ─────────────────
features_to_plot = [
    'cgpa', 'backlog_count', 'internship_count',
    'project_count', 'leetcode_problems_solved',
    'github_contributions', 'skill_diversity_score', 'certification_count'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

palette = {'FAANG': '#6366f1', 'Mid-tier': '#3b82f6',
           'Mass Recruiter': '#10b981', 'Not Placed': '#ef4444'}

for idx, feat in enumerate(features_to_plot):
    for tier in tier_order:
        data = df[df['company_tier'] == tier][feat]
        axes[idx].hist(data, bins=25, alpha=0.55, label=tier,
                       color=palette[tier], edgecolor='none')
    axes[idx].set_title(feat.replace('_', ' ').title(), fontweight='bold')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('Count')
    if idx == 0:
        axes[idx].legend(fontsize=8)

plt.suptitle('Feature Distributions by Company Tier', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.3  Correlation heatmap ──────────────────────────────────────
num_cols = [
    'cgpa', 'backlog_count', 'internship_count', 'internship_duration_months',
    'project_count', 'project_complexity_score', 'certification_count',
    'skill_diversity_score', 'github_contributions', 'github_repo_count',
    'leetcode_problems_solved', 'leetcode_contest_rating', 'placed'
]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.4  Placement rate by branch ────────────────────────────────
branch_rate = df.groupby('degree_branch')['placed'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#6366f1' if r > 60 else '#3b82f6' if r > 40 else '#f97316' if r > 20 else '#ef4444'
          for r in branch_rate.values]
bars = ax.barh(branch_rate.index, branch_rate.values, color=colors, edgecolor='white')
ax.set_xlabel('Placement Rate (%)')
ax.set_title('Placement Rate by Degree Branch', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, branch_rate.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('plot_branch_placement_rate.png', bbox_inches='tight')
plt.show()

---
## ⚙️ Cell 4 — Feature Engineering & Preprocessing

In [ ]:
# ── 4.1  Encode categorical feature ──────────────────────────────
df_model = df.copy()

# Label-encode degree_branch
le_branch = LabelEncoder()
df_model['branch_encoded'] = le_branch.fit_transform(df_model['degree_branch'])
print('Branch encoding:', dict(zip(le_branch.classes_, le_branch.transform(le_branch.classes_))))

# ── 4.2  Engineer derived features ───────────────────────────────

# Coding strength: composite of LeetCode problems + contest rating
df_model['coding_strength'] = (
    np.clip(df_model['leetcode_problems_solved'] / 500, 0, 1) * 0.6 +
    np.clip(df_model['leetcode_contest_rating']  / 2200, 0, 1) * 0.4
).round(4)

# GitHub activity: composite of contributions + repos
df_model['github_activity'] = (
    np.clip(df_model['github_contributions'] / 600, 0, 1) * 0.65 +
    np.clip(df_model['github_repo_count']     / 30,  0, 1) * 0.35
).round(4)

# Experience score: internship count weighted by duration
df_model['experience_score'] = (
    df_model['internship_count'] * 0.6 +
    np.clip(df_model['internship_duration_months'] / 12, 0, 1) * 0.4
).round(4)

# Academic score: CGPA penalised by backlogs
df_model['academic_score'] = (
    (df_model['cgpa'] / 10) - (df_model['backlog_count'] * 0.04)
).clip(0, 1).round(4)

print('\nDerived features added:')
print(df_model[['coding_strength','github_activity','experience_score','academic_score']].describe().round(3))

In [ ]:
# ── 4.3  Define feature sets ──────────────────────────────────────

# Core raw features
RAW_FEATURES = [
    'cgpa',
    'backlog_count',
    'internship_count',
    'internship_duration_months',
    'project_count',
    'project_complexity_score',
    'certification_count',
    'skill_diversity_score',
    'github_contributions',
    'github_repo_count',
    'leetcode_problems_solved',
    'leetcode_contest_rating',
    'branch_encoded',
]

# Extended features (raw + derived)
ALL_FEATURES = RAW_FEATURES + [
    'coding_strength',
    'github_activity',
    'experience_score',
    'academic_score',
]

# ── BINARY target: placed / not placed
X_bin = df_model[ALL_FEATURES]
y_bin = df_model['placed']

# ── MULTI-CLASS target: company tier
le_tier = LabelEncoder()
y_tier  = le_tier.fit_transform(df_model['company_tier'])
X_tier  = df_model[ALL_FEATURES]

print('Feature matrix shape :', X_bin.shape)
print('Binary target dist   :', dict(y_bin.value_counts()))
print('Tier classes         :', list(le_tier.classes_))
print('Tier target dist     :', dict(zip(le_tier.classes_, np.bincount(y_tier))))

In [ ]:
# ── 4.4  Train / Test split ───────────────────────────────────────
TEST_SIZE   = 0.20
RANDOM_SEED = 42

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin, y_bin, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_bin
)

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_tier, y_tier, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_tier
)

print(f'Binary   → Train: {len(X_train_b):,}  |  Test: {len(X_test_b):,}')
print(f'Tier     → Train: {len(X_train_t):,}  |  Test: {len(X_test_t):,}')

In [ ]:
# ── 4.5  SMOTE — handle class imbalance ──────────────────────────
# Apply only to training set
smote = SMOTE(random_state=RANDOM_SEED)

X_train_b_sm, y_train_b_sm = smote.fit_resample(X_train_b, y_train_b)

print('After SMOTE (binary):')
unique, counts = np.unique(y_train_b_sm, return_counts=True)
for u, c in zip(unique, counts):
    label = 'Placed' if u == 1 else 'Not Placed'
    print(f'  {label}: {c:,}')

# For tier classification
X_train_t_sm, y_train_t_sm = smote.fit_resample(X_train_t, y_train_t)
print('\nAfter SMOTE (tier):')
for cls, cnt in zip(le_tier.classes_, np.bincount(y_train_t_sm)):
    print(f'  {cls}: {cnt:,}')

---
## 🤖 Cell 5 — Model 1: XGBoost — Placement Probability

In [ ]:
# ── 5.1  Train XGBoost classifier ────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.08,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    min_child_weight = 3,
    gamma            = 0.1,
    reg_alpha        = 0.05,
    reg_lambda       = 1.5,
    objective        = 'binary:logistic',
    eval_metric      = 'auc',
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
)

xgb_model.fit(
    X_train_b_sm, y_train_b_sm,
    eval_set         = [(X_test_b, y_test_b)],
    verbose          = 50
)
print('\n✅ XGBoost training complete')

In [ ]:
# ── 5.2  XGBoost evaluation ───────────────────────────────────────
y_pred_b      = xgb_model.predict(X_test_b)
y_prob_b      = xgb_model.predict_proba(X_test_b)[:, 1]

xgb_metrics = {
    'Accuracy'  : accuracy_score(y_test_b, y_pred_b),
    'Precision' : precision_score(y_test_b, y_pred_b),
    'Recall'    : recall_score(y_test_b, y_pred_b),
    'F1-Score'  : f1_score(y_test_b, y_pred_b),
    'ROC-AUC'   : roc_auc_score(y_test_b, y_prob_b),
}

print('─' * 38)
print('  XGBoost — Binary Placement Results')
print('─' * 38)
for k, v in xgb_metrics.items():
    bar = '█' * int(v * 20)
    print(f'  {k:<12}: {v:.4f}  {bar}')
print('─' * 38)

print('\nClassification Report:')
print(classification_report(y_test_b, y_pred_b, target_names=['Not Placed', 'Placed']))

In [ ]:
# ── 5.3  XGBoost — Confusion Matrix + ROC Curve ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_b, y_pred_b)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not Placed', 'Placed'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('XGBoost — Confusion Matrix', fontweight='bold')

# ROC Curve
RocCurveDisplay.from_predictions(y_test_b, y_prob_b, ax=axes[1],
                                  color='#6366f1', lw=2,
                                  name=f"XGBoost (AUC={xgb_metrics['ROC-AUC']:.3f})")
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[1].set_title('ROC Curve — Placement Prediction', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_xgb_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4  Cross-Validation ─────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(xgb_model, X_bin, y_bin, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'5-Fold CV ROC-AUC Scores: {cv_scores.round(4)}')
print(f'Mean  : {cv_scores.mean():.4f}')
print(f'Std   : {cv_scores.std():.4f}')

# Plot CV scores
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, 6), cv_scores, color='#6366f1', edgecolor='white', alpha=0.85)
ax.axhline(cv_scores.mean(), color='#f97316', linestyle='--', lw=2, label=f'Mean={cv_scores.mean():.3f}')
ax.set_xlabel('Fold'); ax.set_ylabel('ROC-AUC')
ax.set_title('5-Fold Cross-Validation — XGBoost', fontweight='bold')
ax.set_ylim(0.5, 1.0); ax.legend()
plt.tight_layout()
plt.savefig('plot_cross_validation.png', bbox_inches='tight')
plt.show()

---
## 🏢 Cell 6 — Model 2: Random Forest — Company Tier Classification

In [ ]:
# ── 6.1  Train Random Forest ─────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = 18,
    min_samples_leaf = 4,
    min_samples_split= 8,
    max_features     = 'sqrt',
    class_weight     = 'balanced',
    random_state     = RANDOM_SEED,
    n_jobs           = -1
)

rf_model.fit(X_train_t_sm, y_train_t_sm)
print('✅ Random Forest training complete')

In [ ]:
# ── 6.2  Random Forest evaluation ────────────────────────────────
y_pred_t = rf_model.predict(X_test_t)

rf_accuracy = accuracy_score(y_test_t, y_pred_t)
rf_f1_macro = f1_score(y_test_t, y_pred_t, average='macro')
rf_f1_wtd   = f1_score(y_test_t, y_pred_t, average='weighted')

print('─' * 42)
print('  Random Forest — Tier Classification')
print('─' * 42)
print(f'  Accuracy        : {rf_accuracy:.4f}')
print(f'  F1 (macro)      : {rf_f1_macro:.4f}')
print(f'  F1 (weighted)   : {rf_f1_wtd:.4f}')
print('─' * 42)
print()
print(classification_report(y_test_t, y_pred_t, target_names=le_tier.classes_))

In [ ]:
# ── 6.3  Confusion Matrix — Tier ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
cm_tier = confusion_matrix(y_test_t, y_pred_t)
disp    = ConfusionMatrixDisplay(cm_tier, display_labels=le_tier.classes_)
disp.plot(ax=ax, cmap='Purples', colorbar=False, xticks_rotation=30)
ax.set_title('Random Forest — Company Tier Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_rf_confusion_matrix.png', bbox_inches='tight')
plt.show()

---
## 📊 Cell 7 — Model Comparison: XGBoost vs Logistic Regression vs Random Forest (Binary)

In [ ]:
# ── 7.1  Train baseline models ────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_b_sm)
X_test_scaled  = scaler.transform(X_test_b)

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, n_jobs=-1)
lr_model.fit(X_train_scaled, y_train_b_sm)

rf_bin_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf_bin_model.fit(X_train_b_sm, y_train_b_sm)

# ── 7.2  Compare all models ───────────────────────────────────────
models_eval = {
    'Logistic Regression': (
        lr_model.predict(X_test_scaled),
        lr_model.predict_proba(X_test_scaled)[:, 1]
    ),
    'Random Forest (bin)': (
        rf_bin_model.predict(X_test_b),
        rf_bin_model.predict_proba(X_test_b)[:, 1]
    ),
    'XGBoost': (
        y_pred_b, y_prob_b
    ),
}

comparison = []
for name, (pred, prob) in models_eval.items():
    comparison.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test_b, pred), 4),
        'Precision': round(precision_score(y_test_b, pred), 4),
        'Recall'   : round(recall_score(y_test_b, pred), 4),
        'F1-Score' : round(f1_score(y_test_b, pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_test_b, prob), 4),
})

comp_df = pd.DataFrame(comparison).set_index('Model')
print(comp_df.to_string())

# ── 7.3  Visual comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
x     = np.arange(len(comp_df.columns))
width = 0.25
colors_m = ['#94a3b8', '#10b981', '#6366f1']

for i, (model, row) in enumerate(comp_df.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=model,
                  color=colors_m[i], edgecolor='white', alpha=0.9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width); ax.set_xticklabels(comp_df.columns)
ax.set_ylim(0.5, 1.05); ax.set_ylabel('Score')
ax.set_title('Model Comparison — Binary Placement Classification', fontweight='bold')
ax.legend(); ax.axhline(1.0, color='gray', linestyle=':', lw=0.8)
plt.tight_layout()
plt.savefig('plot_model_comparison.png', bbox_inches='tight')
plt.show()

---
## 🔍 Cell 8 — SHAP Explainability

In [ ]:
# ── 8.1  Compute SHAP values ──────────────────────────────────────
print('Computing SHAP values (may take ~30 seconds)...')

explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test_b)

print(f'SHAP values shape : {shap_vals.shape}')
print('✅ SHAP computation complete')

In [ ]:
# ── 8.2  Global Feature Importance — SHAP Summary ────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_vals, X_test_b,
    feature_names = ALL_FEATURES,
    show          = False,
    plot_size     = (10, 7)
)
plt.title('SHAP Summary — Feature Impact on Placement Probability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.3  SHAP Bar Plot — Mean Absolute Impact ────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_vals, X_test_b,
    feature_names = ALL_FEATURES,
    plot_type     = 'bar',
    show          = False
)
plt.title('SHAP — Mean Absolute Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_shap_importance_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.4  SHAP Waterfall — Individual Student Explanation ─────────
# Show explanation for a specific student (index 0 by default)

STUDENT_IDX = 0   # ← change to any test-set index

shap_explanation = shap.Explanation(
    values        = shap_vals[STUDENT_IDX],
    base_values   = explainer.expected_value,
    data          = X_test_b.iloc[STUDENT_IDX].values,
    feature_names = ALL_FEATURES
)

prob = xgb_model.predict_proba(X_test_b.iloc[[STUDENT_IDX]])[0][1]
actual = y_test_b.iloc[STUDENT_IDX]

print(f'Student #{STUDENT_IDX}')
print(f'  Predicted probability : {prob*100:.1f}%')
print(f'  Actual placement      : {"Placed" if actual==1 else "Not Placed"}')
print(f'  Profile:')
for feat in ALL_FEATURES:
    print(f'    {feat:<30}: {X_test_b.iloc[STUDENT_IDX][feat]}')

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation, show=False)
plt.title(f'SHAP Waterfall — Student #{STUDENT_IDX}  |  P(Placed) = {prob*100:.1f}%',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_shap_waterfall.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.5  GAP ANALYSIS FUNCTION ───────────────────────────────────
# This is what your FastAPI /predict endpoint will call

def analyze_student_gaps(student_features: dict, top_n: int = 5) -> dict:
    """
    Given a student's feature dict, returns:
      - placement probability
      - predicted company tier
      - SHAP-based gap analysis (what's helping / hurting)
      - priority improvement recommendations
    """
    # Build feature row
    student_df = pd.DataFrame([student_features])[ALL_FEATURES]

    # Predictions
    prob      = float(xgb_model.predict_proba(student_df)[0][1])
    tier_enc  = rf_model.predict(student_df)[0]
    tier      = le_tier.inverse_transform([tier_enc])[0]

    # SHAP
    sv = explainer.shap_values(student_df)[0]

    impacts = pd.DataFrame({
        'feature': ALL_FEATURES,
        'shap_value': sv,
        'feature_value': student_df.iloc[0].values
    }).sort_values('shap_value')

    gaps     = impacts[impacts['shap_value'] < 0].head(top_n)
    strengths= impacts[impacts['shap_value'] > 0].tail(top_n)

    # Rule-based recommendations
    recommendations = []
    f = student_features

    if f.get('leetcode_problems_solved', 0) < 100:
        recommendations.append({'priority': 'HIGH',   'action': 'Solve at least 150 LeetCode problems (focus on Easy + Medium)'})
    if f.get('internship_count', 0) == 0:
        recommendations.append({'priority': 'HIGH',   'action': 'Apply to at least 1 internship (even 1-2 months counts)'})
    if f.get('github_contributions', 0) < 50:
        recommendations.append({'priority': 'HIGH',   'action': 'Push at least 3–5 projects to GitHub with proper READMEs'})
    if f.get('project_count', 0) < 2:
        recommendations.append({'priority': 'HIGH',   'action': 'Build 2 more end-to-end projects with deployment'})
    if f.get('backlog_count', 0) > 2:
        recommendations.append({'priority': 'HIGH',   'action': 'Clear pending backlogs — they filter you at resume screening'})
    if f.get('certification_count', 0) < 2:
        recommendations.append({'priority': 'MEDIUM', 'action': 'Complete 2 domain certifications (AWS, Google, Coursera)'})
    if f.get('skill_diversity_score', 0) < 4:
        recommendations.append({'priority': 'MEDIUM', 'action': 'Expand skill set: add 1 backend + 1 cloud + 1 database technology'})
    if f.get('project_complexity_score', 0) < 4:
        recommendations.append({'priority': 'MEDIUM', 'action': 'Upgrade projects: add REST APIs, deployment, or ML components'})

    return {
        'placement_probability_pct': round(prob * 100, 1),
        'predicted_company_tier'   : tier,
        'strengths': strengths[['feature','shap_value']].to_dict('records'),
        'gaps'     : gaps[['feature','shap_value']].to_dict('records'),
        'recommendations': recommendations
    }


# ── Demo: analyze a sample student ───────────────────────────────
sample_student = {
    'cgpa': 7.4,
    'backlog_count': 1,
    'internship_count': 1,
    'internship_duration_months': 2,
    'project_count': 2,
    'project_complexity_score': 4.2,
    'certification_count': 2,
    'skill_diversity_score': 4.5,
    'github_contributions': 95,
    'github_repo_count': 7,
    'leetcode_problems_solved': 60,
    'leetcode_contest_rating': 1350,
    'branch_encoded': int(le_branch.transform(['CS'])[0]),
    'coding_strength': round(min(60/500, 1)*0.6 + min(1350/2200, 1)*0.4, 4),
    'github_activity': round(min(95/600, 1)*0.65 + min(7/30, 1)*0.35, 4),
    'experience_score': round(1*0.6 + min(2/12, 1)*0.4, 4),
    'academic_score': round(7.4/10 - 1*0.04, 4),
}

result = analyze_student_gaps(sample_student)

print(f"\n{'='*50}")
print(f"  STUDENT GAP ANALYSIS REPORT")
print(f"{'='*50}")
print(f"  Placement Probability : {result['placement_probability_pct']}%")
print(f"  Predicted Tier        : {result['predicted_company_tier']}")
print(f"\n  TOP STRENGTHS:")
for s in result['strengths']:
    print(f"    ✅ {s['feature']:<30} +{s['shap_value']:.4f}")
print(f"\n  TOP GAPS (what's hurting you):")
for g in result['gaps']:
    print(f"    ❌ {g['feature']:<30} {g['shap_value']:.4f}")
print(f"\n  RECOMMENDATIONS:")
for r in result['recommendations']:
    icon = '🔴' if r['priority']=='HIGH' else '🟡'
    print(f"    {icon} [{r['priority']}] {r['action']}")

---
## 💾 Cell 9 — Save Models for FastAPI

In [ ]:
# ── 9.1  Save all models & encoders ──────────────────────────────
os.makedirs('models', exist_ok=True)

joblib.dump(xgb_model,    'models/xgb_placement.pkl')
joblib.dump(rf_model,     'models/rf_company_tier.pkl')
joblib.dump(le_branch,    'models/encoder_branch.pkl')
joblib.dump(le_tier,      'models/encoder_tier.pkl')
joblib.dump(scaler,       'models/scaler.pkl')
joblib.dump(explainer,    'models/shap_explainer.pkl')
joblib.dump(ALL_FEATURES, 'models/feature_names.pkl')

print('✅ Models saved to /models/')
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}') / 1024
    print(f'   {f:<30} {size:.1f} KB')

In [ ]:
# ── 9.2  Verify saved models load correctly ───────────────────────
xgb_loaded  = joblib.load('models/xgb_placement.pkl')
rf_loaded   = joblib.load('models/rf_company_tier.pkl')
feats_loaded= joblib.load('models/feature_names.pkl')

test_prob = xgb_loaded.predict_proba(X_test_b[:5])[:, 1]
print('✅ Models reload verified')
print(f'   Sample predictions (probability): {test_prob.round(3)}')

---
## 📋 Cell 10 — Final Summary Report

In [ ]:
print('=' * 58)
print('  PLACEMENT PREDICTOR — MODEL TRAINING SUMMARY')
print('=' * 58)

print(f'\n  Dataset             : {len(df):,} students')
print(f'  Features used       : {len(ALL_FEATURES)}')
print(f'  Train / Test split  : {int((1-TEST_SIZE)*100)}% / {int(TEST_SIZE*100)}%')

print(f'\n  ── XGBoost (Placement Probability) ──────────')
for k, v in xgb_metrics.items():
    print(f'     {k:<14}: {v:.4f}')

print(f'\n  ── Random Forest (Company Tier) ─────────────')
print(f'     Accuracy       : {rf_accuracy:.4f}')
print(f'     F1 (weighted)  : {rf_f1_wtd:.4f}')
print(f'     F1 (macro)     : {rf_f1_macro:.4f}')

print(f'\n  ── Cross-Validation (XGBoost) ───────────────')
print(f'     CV ROC-AUC     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print(f'\n  ── Saved Models ─────────────────────────────')
print(f'     models/xgb_placement.pkl')
print(f'     models/rf_company_tier.pkl')
print(f'     models/shap_explainer.pkl')
print(f'     models/encoder_branch.pkl')
print(f'     models/encoder_tier.pkl')

print(f'\n  ── FastAPI Integration ──────────────────────')
print(f'     POST /api/predict-placement')
print(f'     → placement_probability_pct')
print(f'     → predicted_company_tier')
print(f'     → shap gaps + strengths')
print(f'     → priority recommendations')
print('=' * 58)